## load environment variable

In [28]:
# to load my groq api key from .env file
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
import os
DB_HOST=os.getenv("DB_HOST")
DB_USER=os.getenv("DB_USER")
DB_PASSWORD=os.getenv("DB_PASSWORD")
DB_NAME=os.getenv("DB_NAME")

# import libararies

In [30]:
from langchain.chat_models import init_chat_model

In [32]:
llm=init_chat_model(model='openai/gpt-oss-120b',model_provider='groq',temperature=0)

In [34]:
from pydantic import BaseModel, Field

In [35]:
class Output(BaseModel):
    type:str=Field(description="""
                   * Type of the output. 
                   * Striclty standardize it to "QUERY" or "REMARK" """)
    value:str=Field(description="""
                    * If type is "QUERY" then provide valid Mysql query.
                    * It the type is "REMARK" then provide the short remark or feedback on the user query.
""")

In [36]:
structured_llm=llm.with_structured_output(Output)

In [42]:
sytesm_prompt="""
- You are a expert mysql query generator.
- your task is to analyse the user request and if it is related to table schema defined below then generate valid mysql query without any special character or line seperators.
- If the user query is not related to the define schema then provide short remark.

* Table description:
* Table name: students
* columns: 
    id- unique identifier of the student.
    name- name of the student.
    gender-gender of the student.
    city- city where the student lives.
    age- age of the student.
    marks-marks of the student.
    birthdate- birthdate of the student.

user question is below:
"""

In [71]:
user_query="provide me city wise count of students"
prompt=sytesm_prompt+user_query.lower().strip()

In [72]:
response=structured_llm.invoke(prompt)

In [73]:
response

Output(type='QUERY', value='SELECT city, COUNT(*) AS student_count FROM students GROUP BY city')

In [74]:
from mysql import connector

In [75]:
connection=connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

In [76]:
cursor=connection.cursor()

In [77]:
cursor.execute(response.value)
rows=cursor.fetchall()

In [78]:
rows

[('Mumbai', 6),
 ('Delhi', 3),
 ('Pune', 4),
 ('Chennai', 3),
 ('Kolkata', 2),
 ('Kochi', 2),
 ('Agra', 2),
 ('Jaipur', 2),
 ('Hyderabad', 2),
 ('Surat', 2),
 ('Lucknow', 1),
 ('Ahmedabad', 1),
 ('Nagpur', 1),
 ('Mumnbai', 1)]

In [79]:
output_system_prompt="""
* You are a expert data presenter.
* You will be provided with result of a mysql query.
* Present that data in a user friendly format using correct visualization.
- dont suggest any additional things. just provide the visualization or representation of the data.

* the mysql query result is provided below:
"""

In [83]:
output_prompt=output_system_prompt+str(rows)

In [84]:
output_response=llm.invoke(output_prompt)

In [85]:
print(output_response.content)

**City – Count Overview**

| City      | Count |
|-----------|-------|
| Mumbai    | 6 |
| Delhi     | 3 |
| Pune      | 4 |
| Chennai   | 3 |
| Kolkata   | 2 |
| Kochi     | 2 |
| Agra      | 2 |
| Jaipur    | 2 |
| Hyderabad | 2 |
| Surat     | 2 |
| Lucknow   | 1 |
| Ahmedabad | 1 |
| Nagpur    | 1 |
| Mumnbai   | 1 |

---

### Horizontal Bar Chart  

*(Each “█” represents one unit)*  

```
Mumbai    ██████ (6)
Pune      ████   (4)
Delhi     ███    (3)
Chennai   ███    (3)
Kolkata   ██     (2)
Kochi     ██     (2)
Agra      ██     (2)
Jaipur    ██     (2)
Hyderabad ██     (2)
Surat     ██     (2)
Lucknow   █      (1)
Ahmedabad █      (1)
Nagpur    █      (1)
Mumnbai   █      (1)
```


In [9]:
from pydantic import BaseModel, Field

class Output(BaseModel):
    type: str = Field(..., description="""Type of output. standarsize it to "QUERY" or "REMARK" """)
    value: str = Field(..., description="The content of the output. For QUERY, it should be a valid MySQL query. For REMARK, it should be a short comment.") 
    

In [33]:
structured_llm=llm.with_structured_output(Output)

In [11]:
prompt="""
* you are a expert Mysql query generator. 
* you will be given you a question, and you need to generate a valid, clean MySQL query to answer the question. do not include any line seperars.
* If the question is not clear or cannot be answered with a MySQL query, please provide a remark instead.

* Table description:
* Table name: students
* columns: 
    id- unique identifier of the student.
    name- name of the student.
    gender-gender of the student.
    city- city where the student lives.
    age- age of the student.
    marks-marks of the student.
    birthdate- birthdate of the student.

user question is below:
"""


In [14]:
user_query="What is the average marks of students in each city?"
user_query=user_query.lower().strip()
final_prompt=prompt+user_query
print(final_prompt)


* you are a expert Mysql query generator. 
* you will be given you a question, and you need to generate a valid, clean MySQL query to answer the question. do not include any line seperars.
* If the question is not clear or cannot be answered with a MySQL query, please provide a remark instead.

* Table description:
* Table name: students
* columns: 
    id- unique identifier of the student.
    name- name of the student.
    gender-gender of the student.
    city- city where the student lives.
    age- age of the student.
    marks-marks of the student.
    birthdate- birthdate of the student.

user question is below:
what is the average marks of students in each city?


In [15]:
response=structured_llm.invoke(final_prompt)

In [16]:
response.type
response.value

'SELECT city, AVG(marks) AS average_marks FROM students GROUP BY city'

In [17]:
from mysql import connector

In [18]:
connection=connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME)

In [19]:
cursor=connection.cursor()

In [20]:
cursor.execute(response.value)
rows=cursor.fetchall()

In [21]:
rows

[('Mumbai', 79.38333320617676),
 ('Delhi', 76.1999994913737),
 ('Pune', 80.00000190734863),
 ('Chennai', 91.43333435058594),
 ('Kolkata', 70.30000114440918),
 ('Kochi', 81.8499984741211),
 ('Agra', 73.29999923706055),
 ('Jaipur', 75.45000076293945),
 ('Hyderabad', 70.45000076293945),
 ('Surat', 81.45000076293945),
 ('Lucknow', 63.70000076293945),
 ('Ahmedabad', 68.9000015258789),
 ('Nagpur', 93.0999984741211),
 ('Mumnbai', 100.0)]

In [22]:
USER_OUTPUT_PROMPT="""
* Your are a expert data presenter.
you will be given a result of a MySQL query, and you need to present the result in a clear and concise way.
* use suitable visualization if necessary, but do not include any line seperators in your output.
make it short and clear, and make sure the output is easy to understand for non-technical users.
Query output as below:"""

In [23]:
USER_OUTPUT_FINAL_PROMPT=USER_OUTPUT_PROMPT+str(rows)

In [26]:
response=llm.invoke(USER_OUTPUT_FINAL_PROMPT)

In [27]:
print(response.content)

**City – Value (rounded to 1 decimal)**  

- Mumnbai – 100.0  
- Nagpur – 93.1  
- Chennai – 91.4  
- Pune – 80.0  
- Kochi – 81.8  
- Surat – 81.5  
- Mumbai – 79.4  
- Jaipur – 75.5  
- Delhi – 76.2  
- Agra – 73.3  
- Hyderabad – 70.5  
- Kolkata – 70.3  
- Ahmedabad – 68.9  
- Lucknow – 63.7  

*The highest value is 100 (Mumnbai) and the lowest is 63.7 (Lucknow). The list is ordered from highest to lowest for easy comparison.*


# FEW SHOT EXAMPLE

In [ ]:
prompt="""please provide sentiment of the below feedback.
feedback: i am very afraid.

examples:
feedback: i am very sad with the services.
sentiment: negative

feedback: i am very happy with the services.
sentiment: positive

feedback: i am ok with the services.
sentiment: neutral
"""
response=llm.invoke(prompt)

from pprint import pprint
pprint(response.content)

# ROLE PLAY PROMPTS

In [18]:
prompt="""You are a Expert GEN AI engineer having more that 10 years of experience in building production level
gen ai applications.
please give me tips for gen ai developement.
"""
response=llm.invoke(prompt)

from pprint import pprint
print(response.content)

Below is a compact “cheat‑sheet” that captures the most important lessons I’ve learned over a decade of shipping production‑grade generative‑AI (Gen‑AI) systems—from research prototypes to multi‑billion‑dollar SaaS products.  
Feel free to cherry‑pick the items that match your current stage (proof‑of‑concept → MVP → scale‑out) and adapt them to your stack, domain, and team culture.

---

## 1. Foundations – Choose the Right Problem & Scope

| ✅ Good practice | ❌ Common pitfall |
|-----------------|------------------|
| **Start with a clear business/value hypothesis** – “What concrete outcome does the model enable?” (e.g., faster content creation, reduced manual coding, higher conversion). | Building a “cool” model first and trying to retrofit a use‑case later. |
| **Define the output modality early** – text, code, images, audio, multimodal, or structured data? Each has different data pipelines, latency budgets, and evaluation metrics. | Assuming a single model can serve all modalities 

## COT prompts

In [28]:
prompt=""" you are a marketting expert.
please give me 5 best approaches to market a household product. explain each apprach in 2-3 lines only. 
alos provide the best appraoch among given appraches and why
"""
response=llm.invoke(prompt)

from pprint import pprint
print(response.content)

**5 Proven Approaches to Market a Household Product**

| # | Approach | How It Works (2‑3 lines) |
|---|----------|--------------------------|
| 1 | **Influencer Partnerships** | Team up with micro‑ or macro‑influencers who regularly share home‑care tips. Their authentic demos and reviews reach a highly engaged, trust‑based audience. |
| 2 | **Content‑Driven SEO** | Create a blog/video hub with “how‑to” guides, cleaning hacks, and product comparisons. Optimized for long‑tail keywords, it drives organic traffic and positions the brand as an authority. |
| 3 | **In‑Store Sampling & POP Displays** | Place eye‑catching point‑of‑purchase displays with free trial sizes in high‑traffic aisles. Hands‑on experience converts shoppers who are already in a purchase mindset. |
| 4 | **Social‑Commerce Ads** | Run shoppable ads on platforms like Instagram, TikTok, and Facebook that let users add the product to cart without leaving the feed. The frictionless path boosts impulse buys. |
| 5 | **Sustain